# Theater Examples

Examples for affine theater spaces, clipping, and theater resizing.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML

plt.rcParams["animation.html"] = "jshtml"

for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent):
    src_dir = candidate / "src"
    if src_dir.exists():
        sys.path.insert(0, str(src_dir))
        break
else:
    raise RuntimeError("Could not locate the project's src directory.")

from visualizer import (
    Curve,
    CurveStyle,
    Draw,
    DrawTheater,
    Erase,
    EraseTheater,
    FillBetweenArea,
    FillBetween,
    JitterFillBetween,
    Jitter,
    MoveFillBetween,
    MoveTheater,
    Move,
    Parallel,
    Scene,
    Theater,
    Schedule,
    Stress,
)


In [ ]:
# Example 1: theater background, clipping, and linear resizing
x = np.linspace(0.0, 1.0, 250)
y_local = 1.12 * np.sin(np.pi * x)
y_local = np.maximum(y_local, 0.0)

theater = Theater(
    "panel",
    xlim=(0.18, 0.56),
    ylim=(0.18, 0.56),
    local_xlim=(0.0, 1.0),
    local_ylim=(0.0, 1.0),
    facecolor="#dbeafe",
    edgecolor="#2563eb",
    alpha=0.24,
    linewidth=2.0,
)

schedule = Schedule()
schedule.add(DrawTheater(theater), duration=0.45)
schedule.add(
    Parallel(
        (
            Draw(
                Curve(
                    "theater_curve",
                    x,
                    y_local,
                    theater_id="panel",
                    color="#1d4ed8",
                    linewidth=3.0,
                )
            ),
            FillBetween(
                FillBetweenArea(
                    "theater_fill",
                    x,
                    y_local,
                    0.0,
                    theater_id="panel",
                    color="#93c5fd",
                    alpha=0.35,
                    linewidth=0.0,
                )
            ),
        )
    ),
    duration=1.2,
)
schedule.add(
    MoveTheater(
        "panel",
        xlim=(0.05, 0.82),
        ylim=(0.15, 0.86),
        facecolor="#ede9fe",
        edgecolor="#7c3aed",
        alpha=0.2,
        linewidth=2.5,
    ),
    duration=1.15,
)
schedule.pause(0.35)
schedule.add(Move("theater_curve", x_prime=None, y_prime=0.45 + 0.35 * np.sin(2.0 * np.pi * x), theater_id=None, color="#7c3aed"), duration=0.0)
schedule.add(
    MoveFillBetween(
        "theater_fill",
        x_prime=None,
        y1_prime=0.45 + 0.35 * np.sin(2.0 * np.pi * x),
        y2_prime=np.zeros_like(x),
        theater_id=None,
        color="#c4b5fd",
        alpha=0.28,
    ),
    duration=0.0,
)
schedule.add(EraseTheater("panel"), duration=0.45)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.grid(True, alpha=0.2)
anim = schedule.build_animation(fig=fig, ax=ax, fps=30, title="Theater transform and clipping")
plt.close(fig)
HTML(anim.to_jshtml())
